#### Install deps (only needed in notebook)
#### !pip install -q rdflib requests tqdm python-dotenv

In [2]:
import json
import os
import re
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Dict, Tuple, List

import requests
from dotenv import load_dotenv
from rdflib import (
    Graph,
    RDF,
    RDFS,
    OWL,
    URIRef,
    Literal,
    Namespace,
)
from rdflib.term import BNode
from tqdm import tqdm

### ENV + CONFIG

In [3]:
load_dotenv("bot.env")

API_URL     = os.getenv("WB_API_URL")
USER        = os.getenv("WB_USERNAME")
PASSWORD    = os.getenv("WB_PASSWORD")
SPARQL_URL  = os.getenv("WB_SPARQL_URL")  # optional but needed for true dedup

WB_LANG = "en"

if not API_URL or not USER or not PASSWORD:
    raise RuntimeError("Missing WB_API_URL / WB_USERNAME / WB_PASSWORD in bot.env")

# Property IDs in your instance (MANUALLY created)
PROPS: Dict[str, str] = {
    "orkg_id": "P1",
    "source": "P2",
    "wikidata_uri": "P3",       # URL or QID-as-string

    "has_data_format_specification": "P4",
    "has_data_item": "P5",
    "has_data_model": "P6",
    "has_process": "P7",
    "has_software": "P8",
    "mentions": "P9",
    "has_following_step": "P10",
    "part_of": "P11",
    "has_vmodel_step": "P12",
    "has_preceding_step": "P13",
    "has_parts": "P14",
    "has_corresponding_step": "P15",

    "instance_of": "P16",       # Item-type
    "ontology_iri": "P17",      # String or URL
    "subclass_of": "P18",
}

TOP_THING_LABEL = "Thing (OWL)"
TOP_THING_DESC  = "Top of OWL class hierarchy (owl:Thing)"
ALIAS_KEYS = {"alias", "aliase", "plural"}   # accept typo 'aliase'

# If later you want to map object properties -> PROPS keys, configure here:
PREDICATE_TO_PROPSKEY: Dict[str, str] = {
    # "has_data_model" : "has_data_model",
    # "has_process"    : "has_process",
    # ...
}

### Utility: datatypes / coercion

In [4]:
_QID_RE     = re.compile(r"^Q[1-9]\d*$")
_WD_URL_RE  = re.compile(r"^https?://(www\.)?wikidata\.org/(entity|wiki)/(Q[1-9]\d*)/?$", re.I)

def normalize_wd(value: str) -> Optional[str]:
    """Normalize Wikidata value: accept QID or URL, return bare QID or None."""
    if not value:
        return None
    v = str(value).strip()
    if v in {"None", "null", ""}:
        return None
    if _QID_RE.match(v):
        return v
    m = _WD_URL_RE.match(v)
    return m.group(3) if m else None

def looks_like_url(v: str) -> bool:
    return (v or "").strip().lower().startswith(("http://", "https://"))

def coerce_value_for_prop(pid_key: str, value: str) -> str:
    """
    Coerce Python-side value for a given PROPS key.
    Currently only special-cases wikidata_uri: if a bare QID, convert to URL.
    """
    if pid_key == "wikidata_uri" and value and value.startswith("Q"):
        return f"https://www.wikidata.org/entity/{value}"
    return value

def local_name(uri: URIRef) -> str:
    s = str(uri)
    return s.rsplit("#", 1)[-1].rsplit("/", 1)[-1]

def better_local_name(uri: URIRef) -> str:
    """More robust local name extraction for labels."""
    s = str(uri)
    parts = re.split(r"[#/]", s)
    parts = [p for p in parts if p]
    return parts[-1] if parts else s

def norm_pred_name(name: str) -> str:
    """
    Normalize predicate local name to snake_case:
    hasDataModel -> has_data_model
    """
    s1 = re.sub(r"(.)([A-Z][a-z]+)", r"\1_\2", name)
    s2 = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s1)
    return s2.lower()

### RDF helpers (labels, IRIs, etc.)

In [5]:
SKOS = Namespace("http://www.w3.org/2004/02/skos/core#")

def normalize_iri(u: URIRef) -> URIRef:
    """Fix occasional '#http...' glitches."""
    s = str(u)
    if s.startswith("#http"):
        s = s[1:]
    return URIRef(s)

def is_valid_uriref(u: URIRef) -> bool:
    """Conservative: accept only http/https IRIs."""
    s = str(u)
    return s.startswith(("http://", "https://"))

STRUCTURAL_TYPES = {
    OWL.Class,
    OWL.ObjectProperty,
    OWL.DatatypeProperty,
    OWL.AnnotationProperty,
    OWL.Ontology,
}

def is_individual(g: Graph, s: URIRef) -> bool:
    """
    Heuristic: treat as individual if it has any rdf:type that is not purely structural.
    """
    types = set(g.objects(s, RDF.type))
    if not types:
        return True
    return any(t not in STRUCTURAL_TYPES for t in types)

def first_label(g: Graph, s: URIRef, pref_langs=("en", "de", None)) -> Optional[str]:
    """
    Prefer skos:prefLabel, then rdfs:label, picking by language preference.
    """
    candidates: List[Literal] = []
    for p in (SKOS.prefLabel, RDFS.label):
        for o in g.objects(s, p):
            if isinstance(o, Literal):
                candidates.append(o)
    if not candidates:
        return None

    for lang in pref_langs:
        for o in candidates:
            if lang is None:
                if not o.language:
                    return str(o)
            else:
                if o.language and o.language.lower().startswith(lang):
                    return str(o)

    return str(sorted((str(o) for o in candidates))[0])

def safe_label(g: Graph, s: URIRef) -> str:
    """Use first_label if available, otherwise derive from URI."""
    lbl = first_label(g, s)
    if lbl:
        return lbl
    txt = str(s).split("#")[-1].split("/")[-1]
    return txt or str(s)

### Annotation helpers (aliases, description, IDs)

In [6]:
def sanitize_annotations(raw: Dict[str, List[str]]) -> Dict[str, List[str]]:
    """
    Normalise and split out alias-like keys vs. ID-like etc.
    """
    clean: Dict[str, List[str]] = {}
    for k, vals in (raw or {}).items():
        out: List[str] = []
        for v in vals:
            v = (v or "").strip()
            if not v:
                continue
            if k in ALIAS_KEYS:
                if not looks_like_url(v):
                    out.append(v)
                else:
                    clean.setdefault("wikidata_uri", []).append(v)
            else:
                out.append(v)
        if out:
            clean[k] = out
    return clean

def build_fields(ann: Dict[str, List[str]]) -> Dict[str, Optional[str | List[str]]]:
    """
    Build a small structured view:
    label, description, aliases, wikidata_qid
    """
    label = (ann.get("label") or ann.get("rdfs:label") or [""])[0].strip()
    desc  = (ann.get("description") or ann.get("rdfs:comment") or [""])[0].strip()
    aliases = sorted(
        {v.strip() for k in ALIAS_KEYS for v in ann.get(k, []) if v and not looks_like_url(v)}
    )
    wd_q = None
    for key in ("wikidata_uri", "wikidata", "wd", "wikidata link"):
        for v in ann.get(key, []):
            q = normalize_wd(v)
            if q:
                wd_q = q
                break
        if wd_q:
            break
    return {
        "label": label,
        "description": desc,
        "aliases": aliases,
        "wikidata_qid": wd_q,
    }

def collect_annotations_min(g: Graph, s: URIRef) -> Dict[str, List[str]]:
    """
    Minimal heuristic: collect a primary label and some known literal annotations.
    """
    lbl = first_label(g, s) or local_name(s)
    ann: Dict[str, List[str]] = {"label": [lbl]}
    for p, o in g.predicate_objects(s):
        if not isinstance(o, Literal) or not isinstance(p, URIRef):
            continue
        pn = local_name(p).lower()
        if pn in {
            "aliase",
            "alias",
            "plural",
            "source",
            "wikidata_uri",
            "orkg_id",
            "description",
            "rdfs:comment",
            "comment",
        }:
            ann.setdefault(pn, []).append(str(o))
    return ann

### Wikibase client with dynamic timeout + throttle

In [7]:
class WBClient:
    _QID_RE = re.compile(r"^Q([1-9]\d*)$")

    def __init__(
        self,
        api_url: str,
        username: str,
        password: str,
        user_agent: str = "OWL2WB/1.0",
        dry_run: bool = False,
        timeout: float = 10.0,
        max_retries: int = 3,
        sleep_between: float = 0.2,
    ):
        self.api_url = api_url.rstrip("?")
        self.s = requests.Session()
        self.s.headers["User-Agent"] = user_agent
        self.username = username
        self.password = password
        self.csrf: Optional[str] = None
        self.dry = dry_run

        # rate + reliability controls
        self.base_timeout = timeout
        self.max_retries = max_retries
        self.sleep_between = sleep_between

    # ---- internal HTTP helpers with retry/backoff ----

    def _request(self, method: str, *, params=None, data=None) -> dict:
        """HTTP wrapper with exponential backoff + timeout."""
        last_exc: Optional[Exception] = None
        for attempt in range(self.max_retries):
            timeout = self.base_timeout * (2 ** attempt)
            try:
                if method == "GET":
                    r = self.s.get(self.api_url, params=params, timeout=timeout)
                else:
                    r = self.s.post(self.api_url, data=data, timeout=timeout)
                if r.status_code >= 500 or r.status_code == 429:
                    # server overloaded or rate-limited
                    time.sleep(self.sleep_between * (attempt + 1))
                    continue
                r.raise_for_status()
                time.sleep(self.sleep_between)
                j = r.json()
                if "error" in j:
                    raise RuntimeError(f"WB API error: {j['error']}")
                return j
            except Exception as e:
                last_exc = e
                time.sleep(self.sleep_between * (attempt + 1))
                continue
        raise RuntimeError(f"Request failed after {self.max_retries} attempts: {last_exc}")

    def _get(self, p: dict) -> dict:
        return self._request("GET", params=p)

    def _post(self, d: dict) -> dict:
        # skip writes in dry-run mode
        if self.dry and d.get("action") in {"wbeditentity", "wbcreateclaim", "wbsetaliases"}:
            print(f"[DRY-RUN] {d.get('action')} -> {json.dumps({k: v for k, v in d.items() if k != 'token'})}")
            return {"dry-run": True}
        return self._request("POST", data=d)

    # ---- SPARQL ----

    def sparql_select(self, query: str) -> List[dict]:
        if not SPARQL_URL:
            return []
        last_exc: Optional[Exception] = None
        for attempt in range(self.max_retries):
            timeout = self.base_timeout * (2 ** attempt)
            try:
                r = self.s.get(
                    SPARQL_URL,
                    params={"query": query, "format": "json"},
                    timeout=timeout,
                )
                if r.status_code >= 500 or r.status_code == 429:
                    time.sleep(self.sleep_between * (attempt + 1))
                    continue
                r.raise_for_status()
                time.sleep(self.sleep_between)
                j = r.json()
                return j.get("results", {}).get("bindings", [])
            except Exception as e:
                last_exc = e
                time.sleep(self.sleep_between * (attempt + 1))
                continue
        raise RuntimeError(f"SPARQL failed after {self.max_retries} attempts: {last_exc}")

    # ---- login ----

    def login(self):
        if self.dry:
            self.csrf = "DRYRUN"
            print("DRY-RUN login (no real login performed).")
            return
        # login token
        t = self._get({
            "action": "query",
            "meta": "tokens",
            "type": "login",
            "format": "json",
        })["query"]["tokens"]["logintoken"]
        res = self._post({
            "action": "login",
            "lgname": self.username,
            "lgpassword": self.password,
            "lgtoken": t,
            "format": "json",
        })
        assert res.get("login", {}).get("result") == "Success", res
        t = self._get({
            "action": "query",
            "meta": "tokens",
            "type": "csrf",
            "format": "json",
        })
        self.csrf = t["query"]["tokens"]["csrftoken"]

    # ---- entity helpers ----

    def search_item_by_label(self, label: str, lang: str = WB_LANG) -> Optional[str]:
        r = self._get({
            "action": "wbsearchentities",
            "search": label,
            "language": lang,
            "type": "item",
            "format": "json",
        })
        for hit in r.get("search", []):
            if hit.get("label") == label and hit.get("id", "").startswith("Q"):
                return hit["id"]
        return None

    def get_entity(self, qid: str) -> dict:
        return self._get({
            "action": "wbgetentities",
            "ids": qid,
            "format": "json",
        })["entities"][qid]

    def get_or_create_item_by_label(
        self,
        label: str,
        description: Optional[str] = None,
        lang: str = WB_LANG,
    ) -> str:
        qid = self.search_item_by_label(label, lang=lang)
        if qid:
            return qid
        if not label or looks_like_url(label):
            raise ValueError(f"Refusing to create item with invalid label: {label!r}")
        data: dict = {"labels": {lang: {"language": lang, "value": label}}}
        if description:
            data["descriptions"] = {lang: {"language": lang, "value": description}}
        res = self._post({
            "action": "wbeditentity",
            "new": "item",
            "data": json.dumps(data),
            "token": self.csrf,
            "format": "json",
        })
        ent = res.get("entity")
        if not ent:
            raise RuntimeError(f"wbeditentity returned no entity. Response: {res}")
        return ent["id"]

    @staticmethod
    def _wb_value_item(qid: str) -> dict:
        m = WBClient._QID_RE.match(qid)
        if not m:
            raise ValueError(f"Not a QID: {qid}")
        return {
            "entity-type": "item",
            "numeric-id": int(m.group(1)),
            "id": qid,
        }

    def add_item_statement(self, qid: str, pid: str, target_qid: str):
        ent = self.get_entity(qid)
        for cl in ent.get("claims", {}).get(pid, []):
            val = cl.get("mainsnak", {}).get("datavalue", {}).get("value", {})
            if val.get("id") == target_qid:
                return
        self._post({
            "action": "wbcreateclaim",
            "entity": qid,
            "property": pid,
            "snaktype": "value",
            "value": json.dumps(self._wb_value_item(target_qid)),
            "token": self.csrf,
            "format": "json",
        })

    def add_string_statement(self, qid: str, pid: str, value: str):
        if not value:
            return
        ent = self.get_entity(qid)
        for cl in ent.get("claims", {}).get(pid, []):
            if str(cl.get("mainsnak", {}).get("datavalue", {}).get("value")) == value:
                return
        self._post({
            "action": "wbcreateclaim",
            "entity": qid,
            "property": pid,
            "snaktype": "value",
            "value": json.dumps(value),
            "token": self.csrf,
            "format": "json",
        })

    def add_aliases(self, qid: str, aliases: List[str], lang: str = WB_LANG):
        if not aliases:
            return
        ent = self.get_entity(qid)
        have = set(a["value"] for a in ent.get("aliases", {}).get(lang, []))
        to_add = [a for a in set(aliases) if a and a not in have]
        if not to_add:
            return
        self._post({
            "action": "wbsetaliases",
            "id": qid,
            "language": lang,
            "add": "|".join(to_add),
            "token": self.csrf,
            "format": "json",
        })


### Dedup helpers: find/get-or-create by ontology_iri

In [8]:
def find_qid_by_property(wb: WBClient, pid: str, value: str) -> Optional[str]:
    """
    Use SPARQL to find an item where wdt:PID == "value".
    Requires WB_SPARQL_URL to be set in env.
    """
    if not SPARQL_URL:
        return None
    q = f"""
    SELECT ?item WHERE {{
      ?item wdt:{pid} "{value}" .
    }} LIMIT 1
    """
    rows = wb.sparql_select(q)
    if not rows:
        return None
    iri = rows[0]["item"]["value"]
    return iri.rsplit("/", 1)[-1]

def get_or_create_by_ontology_iri(
    wb: WBClient,
    subject_iri: str,
    label: str,
    desc: str,
    pid_ontology_iri: str,
    lang: str,
) -> str:
    """
    1. Try to reuse item with ontology_iri == subject_iri.
    2. Else create new item by label/desc.
    3. Ensure ontology_iri statement is present.
    """
    qid = find_qid_by_property(wb, pid_ontology_iri, subject_iri)
    if qid:
        return qid

    qid = wb.get_or_create_item_by_label(label, desc, lang=lang)
    wb.add_string_statement(qid, pid_ontology_iri, subject_iri)
    return qid


### Class pre-pass: create classes + subclass-of hierarchy

In [9]:
def class_prepass(
    g: Graph,
    wb: WBClient,
    rp=None,
    current_file: str = "",
) -> Dict[URIRef, Tuple[str, str]]:
    """
    - Create/lookup items for all OWL/RDFS classes.
    - Anchor owl:Thing to TOP_THING_LABEL.
    - Wire rdfs:subClassOf -> PROPS["subclass_of"].
    Returns: dict[URIRef] -> (label, qid)
    """
    cls_map: Dict[URIRef, Tuple[str, str]] = {}

    # Anchor owl:Thing
    try:
        top_qid = wb.get_or_create_item_by_label(TOP_THING_LABEL, TOP_THING_DESC, lang=WB_LANG)
    except Exception as e:
        top_qid = None
        if rp is not None:
            rp.warnings.append(f"{current_file}: failed to create anchor for owl:Thing → {e}")

    def ensure_class(subject: URIRef) -> Optional[Tuple[str, str]]:
        s = normalize_iri(subject)
        if not is_valid_uriref(s) or isinstance(s, BNode):
            return None
        label = first_label(g, s) or safe_label(g, s)
        try:
            if "ontology_iri" in PROPS:
                qid = get_or_create_by_ontology_iri(
                    wb,
                    str(s),
                    label,
                    "Imported OWL/RDFS class",
                    PROPS["ontology_iri"],
                    WB_LANG,
                )
            else:
                qid = wb.get_or_create_item_by_label(label, "Imported OWL/RDFS class", lang=WB_LANG)
            return (label, qid)
        except Exception as e:
            if rp is not None:
                rp.warnings.append(f"{current_file}: class {s} ({label!r}) create/get failed → {e}")
            return None

    # 1) create/lookup classes
    classes = set(g.subjects(RDF.type, OWL.Class)) | set(g.subjects(RDF.type, RDFS.Class))
    for cls in classes:
        if not isinstance(cls, URIRef):
            continue
        if cls in cls_map:
            continue
        res = ensure_class(cls)
        if res:
            cls_map[normalize_iri(cls)] = res

    if rp is not None:
        rp.classes += len(cls_map)

    # 2) subclass edges
    seen = set()
    for child, parent in g.subject_objects(RDFS.subClassOf):
        if not (isinstance(child, URIRef) and isinstance(parent, URIRef)):
            continue
        child = normalize_iri(child)
        parent = normalize_iri(parent)
        if not is_valid_uriref(child) or not is_valid_uriref(parent):
            continue
        if isinstance(child, BNode) or isinstance(parent, BNode):
            continue

        if child not in cls_map:
            res = ensure_class(child)
            if not res:
                continue
            cls_map[child] = res
        c_label, c_qid = cls_map[child]

        if parent == OWL.Thing and top_qid:
            p_label, p_qid = TOP_THING_LABEL, top_qid
        else:
            if parent not in cls_map:
                res = ensure_class(parent)
                if not res:
                    continue
                cls_map[parent] = res
            p_label, p_qid = cls_map[parent]

        key = (c_qid, p_qid)
        if key in seen:
            continue
        try:
            wb.add_item_statement(c_qid, PROPS["subclass_of"], p_qid)
            seen.add(key)
            if rp is not None:
                rp.subclass_links += 1
        except Exception as e:
            if rp is not None:
                rp.warnings.append(f"{current_file}: subclass link {c_label!r} ⟶ {p_label!r} failed → {e}")

    return cls_map

### Individual import

In [10]:
def process_named_individual(
    g: Graph,
    subj_uri: URIRef,
    cls_map: Dict[URIRef, Tuple[str, str]],
    wb: WBClient,
    rp=None,
    current_file: str = "",
) -> Optional[str]:
    """
    - Creates/updates the item for subj_uri.
    - Sets label/description/aliases.
    - Adds external IDs (Wikidata, ORKG, source).
    - Adds rdf:type as instance_of.
    - Optionally maps selected predicates → PROPS (PREDICATE_TO_PROPSKEY).
    """
    subj_uri = normalize_iri(subj_uri)

    # 1) Gather + clean annotations
    ann = sanitize_annotations(collect_annotations_min(g, subj_uri))
    fields = build_fields(ann)
    label = fields["label"]
    description = fields["description"]
    aliases = fields["aliases"]
    wd_qid = fields["wikidata_qid"]

    if not label:
        if rp is not None:
            rp.warnings.append(f"{current_file}:{subj_uri} → missing label, skipped")
        return None

    # 2) Create / get the item
    qid = wb.get_or_create_item_by_label(label, description or None, lang=WB_LANG)

    # 3) Built-in aliases
    if aliases:
        try:
            wb.add_aliases(qid, aliases, lang=WB_LANG)
            if rp is not None:
                rp.aliases += len(aliases)
        except Exception as e:
            if rp is not None:
                rp.warnings.append(f"{current_file}:{subj_uri} → wbsetaliases failed: {e}")

    # 4) Extra IDs / provenance
    if wd_qid:
        value = coerce_value_for_prop("wikidata_uri", wd_qid)
        wb.add_string_statement(qid, PROPS["wikidata_uri"], value)
        if rp is not None:
            rp.strings += 1

    for s in ann.get("source", []):
        wb.add_string_statement(qid, PROPS["source"], s)
        if rp is not None:
            rp.strings += 1

    for oid in ann.get("orkg_id", []):
        wb.add_string_statement(qid, PROPS["orkg_id"], oid)
        if rp is not None:
            rp.strings += 1

    # 5) rdf:type → instance_of
    for cls in g.objects(subj_uri, RDF.type):
        if cls == OWL.NamedIndividual or not isinstance(cls, URIRef):
            continue
        cls = normalize_iri(cls)
        if cls in cls_map:
            _, class_qid = cls_map[cls]
        else:
            # fallback: create class on the fly
            c_label = first_label(g, cls) or local_name(cls)
            class_qid = wb.get_or_create_item_by_label(c_label, "Imported class (from rdf:type)", lang=WB_LANG)
            cls_map[cls] = (c_label, class_qid)
        wb.add_item_statement(qid, PROPS["instance_of"], class_qid)
        if rp is not None:
            rp.instance_links += 1

    # 6) Selected extra predicates (object properties) → PROPS
    if PREDICATE_TO_PROPSKEY:
        for p, o in g.predicate_objects(subj_uri):
            if not isinstance(p, URIRef):
                continue
            if p in (RDF.type, RDFS.label, RDFS.comment, SKOS.prefLabel, SKOS.altLabel):
                continue

            pn_raw = local_name(p)          # e.g. hasDataModel
            pn = norm_pred_name(pn_raw)     # e.g. has_data_model

            if pn in {
                "alias",
                "aliase",
                "plural",
                "source",
                "wikidata_uri",
                "orkg_id",
                "description",
                "rdfs_comment",
                "comment",
            }:
                continue

            props_key = PREDICATE_TO_PROPSKEY.get(pn)
            if not props_key:
                continue
            pid = PROPS.get(props_key)
            if not pid:
                continue

            if isinstance(o, URIRef):
                if not is_valid_uriref(o):
                    if rp is not None:
                        rp.warnings.append(f"{current_file}:{subj_uri} → invalid URIRef target {o}")
                    continue
                tgt_label = safe_label(g, o)
                if not tgt_label or looks_like_url(tgt_label):
                    if rp is not None:
                        rp.warnings.append(
                            f"{current_file}:{subj_uri} → skip URL-like/empty label {tgt_label!r} from {o}"
                        )
                    continue
                try:
                    tgt_qid = wb.get_or_create_item_by_label(
                        tgt_label,
                        "Imported from object property",
                        lang=WB_LANG,
                    )
                    wb.add_item_statement(qid, pid, tgt_qid)
                except Exception as e:
                    if rp is not None:
                        rp.warnings.append(f"{current_file}:{subj_uri} → target {o} ({tgt_label!r}) failed: {e}")
                    continue

    if rp is not None:
        rp.individuals += 1

    return qid

### Reporting & runner

In [11]:
@dataclass
class Report:
    files: List[str] = field(default_factory=list)
    classes: int = 0
    subclass_links: int = 0
    individuals: int = 0
    instance_links: int = 0
    aliases: int = 0
    strings: int = 0
    warnings: List[str] = field(default_factory=list)

def import_path(
    input_path: str | Path,
    dry_run: bool = False,
    request_timeout: float = 10.0,
    sleep_between: float = 0.2,
) -> Report:
    wb = WBClient(
        API_URL,
        USER,
        PASSWORD,
        dry_run=dry_run,
        timeout=request_timeout,
        sleep_between=sleep_between,
    )
    wb.login()

    rp = Report()
    paths: List[Path] = []
    p = Path(input_path)

    if p.is_dir():
        for f in sorted(p.rglob("*")):
            if f.suffix.lower() in {".owl", ".ttl", ".rdf", ".xml", ".nt", ".n3", ".trig", ".trix"}:
                paths.append(f)
    else:
        paths = [p]

    rp.files = [x.as_posix() for x in paths]

    for f in paths:
        print(f"\n=== {f.name} ===")
        current_file = f.name

        g = Graph()
        g.parse(f.as_posix())

        cls_map = class_prepass(g, wb, rp=rp, current_file=current_file)

        inds = list(g.subjects(RDF.type, OWL.NamedIndividual))
        for s in tqdm(inds, desc="Individuals"):
            try:
                if not isinstance(s, URIRef) or isinstance(s, BNode):
                    continue
                if not is_valid_uriref(s):
                    continue
                process_named_individual(g, s, cls_map, wb, rp=rp, current_file=current_file)
            except Exception as e:
                rp.warnings.append(f"{current_file}:{s} → {e}")

    print("\nDone.")
    return rp

### OpenRefine

In [12]:
import csv

def export_for_openrefine(
    input_path: str | Path,
    output_csv: str | Path,
) -> None:
    """
    Parse OWL/RDF files and export a flat table for OpenRefine.

    - Does NOT talk to Wikibase.
    - Uses the existing annotation helpers to build label/description/aliases.
    - Exports both classes and named individuals.
    """
    p = Path(input_path)
    paths: List[Path] = []

    if p.is_dir():
        for f in sorted(p.rglob("*")):
            if f.suffix.lower() in {".owl", ".ttl", ".rdf", ".xml", ".nt", ".n3", ".trig", ".trix"}:
                paths.append(f)
    else:
        paths = [p]

    rows: List[Dict[str, str]] = []

    for f in paths:
        print(f"\n[EXPORT] Parsing {f.name}")
        g = Graph()
        g.parse(f.as_posix())

        # ---- 1) Classes ----
        classes = set(g.subjects(RDF.type, OWL.Class)) | set(g.subjects(RDF.type, RDFS.Class))
        for cls in classes:
            if not isinstance(cls, URIRef):
                continue
            cls = normalize_iri(cls)
            if not is_valid_uriref(cls):
                continue

            ann = sanitize_annotations(collect_annotations_min(g, cls))
            fields = build_fields(ann)

            super_iris = [
                str(normalize_iri(p_iri)) for p_iri in g.objects(cls, RDFS.subClassOf)
                if isinstance(p_iri, URIRef) and is_valid_uriref(p_iri)
            ]

            rows.append({
                "entity_type": "class",
                "owl_iri": str(cls),
                "label": fields["label"] or better_local_name(cls),
                "description": fields["description"] or "",
                "aliases": "|".join(fields["aliases"] or []),
                "wikidata_qid": fields["wikidata_qid"] or "",
                "super_iris": "|".join(super_iris),
                # this is the value we’ll later store in P17 (ontology_iri)
                "ontology_iri": str(cls),
            })

        # ---- 2) Named individuals ----
        inds = list(g.subjects(RDF.type, OWL.NamedIndividual))
        for s in inds:
            if not isinstance(s, URIRef) or isinstance(s, BNode):
                continue
            s = normalize_iri(s)
            if not is_valid_uriref(s):
                continue

            ann = sanitize_annotations(collect_annotations_min(g, s))
            fields = build_fields(ann)

            class_iris = [
                str(normalize_iri(c)) for c in g.objects(s, RDF.type)
                if isinstance(c, URIRef) and c not in {OWL.NamedIndividual}
            ]

            rows.append({
                "entity_type": "individual",
                "owl_iri": str(s),
                "label": fields["label"] or better_local_name(s),
                "description": fields["description"] or "",
                "aliases": "|".join(fields["aliases"] or []),
                "wikidata_qid": fields["wikidata_qid"] or "",
                "class_iris": "|".join(class_iris),
                "ontology_iri": str(s),
            })

    out_path = Path(output_csv)
    if not rows:
        print("[EXPORT] No rows to write.")
        return

    fieldnames = sorted({k for row in rows for k in row.keys()})
    print(f"\n[EXPORT] Writing {len(rows)} rows to {out_path}")
    with out_path.open("w", newline="", encoding="utf-8") as f_out:
        w = csv.DictWriter(f_out, fieldnames=fieldnames)
        w.writeheader()
        for row in rows:
            w.writerow(row)


In [13]:
export_for_openrefine("slr_reviewed.owl", "aerospace_openrefine_raw.csv")


[EXPORT] Parsing slr_reviewed.owl


wptmp:entity#data importing does not look like a valid URI, trying to serialize this will break.
wptmp:entity#export data does not look like a valid URI, trying to serialize this will break.



[EXPORT] Writing 963 rows to aerospace_openrefine_raw.csv


### Example calls Dry-Run (No upload)

In [14]:
rp = import_path("slr_reviewed.owl", dry_run=True, request_timeout=5.0, sleep_between=0.1)
print(rp)



DRY-RUN login (no real login performed).

=== slr_reviewed.owl ===


wptmp:entity#data importing does not look like a valid URI, trying to serialize this will break.
wptmp:entity#export data does not look like a valid URI, trying to serialize this will break.
Individuals: 100%|████████████████████████████| 956/956 [14:13<00:00,  1.12it/s]


Done.
Report(files=['slr_reviewed.owl'], classes=7, subclass_links=7, individuals=956, instance_links=956, aliases=770, strings=2013, warnings=[])


In [15]:
rp = import_path("slr_reviewed.owl", dry_run=False, request_timeout=10.0, sleep_between=0.2)
print(rp)


=== slr_reviewed.owl ===


wptmp:entity#data importing does not look like a valid URI, trying to serialize this will break.
wptmp:entity#export data does not look like a valid URI, trying to serialize this will break.
Individuals: 100%|████████████████████████████| 956/956 [19:59<00:00,  1.26s/it]


Done.
Report(files=['slr_reviewed.owl'], classes=7, subclass_links=7, individuals=956, instance_links=956, aliases=770, strings=2013, warnings=[])
